# Stance Detection Context Analysis

This notebook compares fake-news and stance predictions on selected examples. The goal is qualitative: identify cases where stance adds context to a misinformation review workflow.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.datasets import load_fake_news_kaggle_bundle, load_stance_detection_bundle
from utils.model_loader import load_fake_news_model, load_stance_model
from utils.prediction import predict_text

pd.set_option("display.max_colwidth", 180)

## Select Examples

The two datasets are not paired by claim. This notebook samples representative fake-news articles and stance headline/body pairs separately, then compares what each model contributes.

In [ ]:
fake_news_bundle = load_fake_news_kaggle_bundle()
stance_bundle = load_stance_detection_bundle()

fake_examples = fake_news_bundle.test.sample(n=5, random_state=42).reset_index(drop=True)
stance_examples = stance_bundle.test.sample(n=5, random_state=42).reset_index(drop=True)

fake_examples[["title", "label"]].head(), stance_examples[["Headline", "label"]].head()

## Run Predictions

In [ ]:
fake_model, fake_tokenizer = load_fake_news_model()
stance_model, stance_tokenizer = load_stance_model()

fake_labels = fake_news_bundle.labels
stance_labels = stance_bundle.labels

def predict_row(model, tokenizer, labels, text):
    predicted_index, probability_vector, prediction_text = predict_text(model, tokenizer, text)
    probabilities = [float(value) for value in probability_vector]
    return {
        "predicted_label": labels[int(predicted_index)],
        "confidence": max(probabilities),
        "prediction_text": prediction_text,
        **{f"prob_{label.lower()}": probabilities[index] for index, label in enumerate(labels)},
    }

fake_outputs = []
for _, row in fake_examples.iterrows():
    output = predict_row(fake_model, fake_tokenizer, fake_labels, row["input_text"])
    output.update({"title": row.get("title", ""), "true_label": row["label"]})
    fake_outputs.append(output)

stance_outputs = []
for _, row in stance_examples.iterrows():
    output = predict_row(stance_model, stance_tokenizer, stance_labels, row["input_text"])
    output.update({"headline": row["Headline"], "true_label": row["label"]})
    stance_outputs.append(output)

fake_output_frame = pd.DataFrame(fake_outputs)
stance_output_frame = pd.DataFrame(stance_outputs)

fake_output_frame[["title", "true_label", "predicted_label", "confidence"]]

In [ ]:
stance_output_frame[["headline", "true_label", "predicted_label", "confidence"]]

## Interpretation Notes

Use this section to record qualitative findings after running the notebook.

- Fake-news prediction answers whether text resembles the model's fake/real training examples.
- Stance prediction answers whether a headline/body pair agrees, disagrees, discusses, or is unrelated.
- Stance can add context, especially when an article discusses a claim without directly supporting it.
- Stance does not verify factual truth and should not be presented as a replacement for fact-checking.